# Geometric programming for inductor design
This notebook aims to replicate the paper titled _Optimization of Inductor Circuits via Geometric Programming_ by Hershenon et al using python.

## Shunt resistance $R_p$
We need to find an expression for $k_6$ so we can express $R_p$ as a monomial.

Starting from the formula for $R_p$
$$
R_p = \frac{1 + [\omega R_\mathrm{si} (C_\mathrm{si} + C_\mathrm{ox}) ]^2}{\omega^2R_\mathrm{si}C_\mathrm{ox}^2}
$$
we substitute the capacitances and resistances for their corresponding monomial expressions
$$
C_\mathrm{ox} = k_2lw
$$
$$
C_\mathrm{si} = k_4lw
$$
$$
R_\mathrm{si} = \frac{k_5}{lw}
$$
For simplicity, let $x=lw$.


In [1]:
import sympy as sp

# Symbols
k2, k4, k5, omeg = sp.symbols('k2 k4 k5 omega', positive=True)
x = sp.symbols('x', positive=True)  # x = l*w

# Monomial models
Cox = k2 * x
Csi = k4 * x
Rsi = k5 / x

# Original Rp expression from the paper
Rp = (1 + (omeg*Rsi*(Csi+Cox))**2)/(omeg**2*Rsi*Cox**2)
# Simplify expression
Rp_simplified = sp.simplify(Rp)

print("Rp simplified:")
sp.pprint(Rp_simplified)

Rp simplified:
  2  2          2    
k₅ ⋅ω ⋅(k₂ + k₄)  + 1
─────────────────────
       2     2       
     k₂ ⋅k₅⋅ω ⋅x     


Now we have
$$
R_p = \frac{k_5^2\omega^2(k_2+k_4)^2+1}{k_2^2k_5\omega^2x}
$$

In [2]:
k6 = sp.simplify(Rp_simplified.coeff(x, -1))
print("k6:")
k6 = sp.simplify(k6)

print(sp.latex(k6))

k6:
\frac{k_{5} \left(k_{2} + k_{4}\right)^{2}}{k_{2}^{2}} + \frac{1}{k_{2}^{2} k_{5} \omega^{2}}


This way we obtain
$$
k_6=
\frac{k_{5} \left(k_{2} + k_{4}\right)^{2}}{k_{2}^{2}} + \frac{1}{k_{2}^{2} k_{5} \omega^{2}}
$$

## Shunt capacitance $C_p$
We need to find expressions for $k_7$ and $k_8$ so we can express $C_p$ as a posynomial.

Starting from the formula for $C_p$
$$
C_p = \frac{C_\mathrm{ox}+\omega^2R_\mathrm{si}^2(C_\mathrm{si} + C_\mathrm{ox})C_\mathrm{si}C_\mathrm{ox}}{1 + [\omega R_\mathrm{si}(C_\mathrm{si} + C_\mathrm{ox})]^2}
$$
_Note: The original formula from the paper contains an error: the $R_\mathrm{si}$ term in the numerator should be squared (as it is here)._

First, substitute the capacitances and resistances by their corresponding monomial expressions
$$
C_\mathrm{ox} = k_2lw
$$
$$
C_\mathrm{si} = k_4lw
$$
$$
R_\mathrm{si} = \frac{k_5}{lw}
$$
For simplicity, let $x=lw$.

In [3]:
# Corrected Cp expression
Cp = (Cox + omeg**2 * Rsi**2 * (Csi + Cox) * Csi * Cox) / (1 + (omeg * Rsi * (Csi + Cox))**2)

# Simplify expression
Cp_simplified = sp.simplify(Cp)

print("Cp simplified:")
sp.pprint(Cp_simplified)

Cp simplified:
     ⎛     2  2              ⎞
k₂⋅x⋅⎝k₄⋅k₅ ⋅ω ⋅(k₂ + k₄) + 1⎠
──────────────────────────────
      2  2          2         
    k₅ ⋅ω ⋅(k₂ + k₄)  + 1     


Now we have
$$
C_p = \frac{k_2 x \left(k_4k_5^2\omega^2(k_2+k_4) + 1\right)}{k_5^2\omega^2(k_2 + k_4)^2 + 1}
$$
and we can treat the expression as a polynomial in $x$ to get the coefficients.

In [4]:
Cp_simplified = sp.simplify(Cp)

k7 = sp.simplify(Cp_simplified / x)

print("k7:")
print(sp.latex(k7))

k7:
\frac{k_{2} \left(k_{4} k_{5}^{2} \omega^{2} \left(k_{2} + k_{4}\right) + 1\right)}{k_{5}^{2} \omega^{2} \left(k_{2} + k_{4}\right)^{2} + 1}


This way we obtain
$$ 
k_7= \frac{k_{2} \left(k_{2} k_{4} k_{5}^{2} \omega^{2} + k_{4}^{2} k_{5}^{2} \omega^{2} + 1\right)}{k_{2}^{2} k_{5}^{2} \omega^{2} + 2 k_{2} k_{4} k_{5}^{2} \omega^{2} + k_{4}^{2} k_{5}^{2} \omega^{2} + 1}
$$

## Q factor
The $Q$ factor is defined as
$$
Q = \frac{\mathrm{Im}(Z)}{\mathrm{Re}(Z)},
$$
where $Z$ is the is the driving point impedance between the terminals. To get this impedance we can express the equivalent circuit as:

<p align="center">
<img src="imag/TwoPort.png">
</p>

The current entering port 1 is
$$
I_1 = V_1 Y_p + Y_s(V_1 - V_2) + \frac{V_1-V_2}{Z_s}
$$
and the current that exits port 2 is
$$
I_2 = V_2Y_p + Y_s(V_2 - V_1) + \frac{V_2-V_1}{Z_s},
$$
thus
$$
V = V_1 - V_2.
$$
And beacuse this is a reciprocal network
$$
V_1 = \frac{V}{2},\qquad V_2 = -\frac{V}{2}
$$
$$
I = I_1 = -I_2.
$$

Now we can write $I_1$ as
$$
I = Y_p\frac{V}{2} + Y_sV + \frac{V}{Z_s},
$$
and find the impedance between terminals
$$
Z = \frac{V}{I} = \frac{1}{\frac{Y_p}{2} + Y_s + \frac{1}{Z_s}}
$$


In [5]:
Rs, Ls, Cs, Rp, Cp = sp.symbols("R_s, L_s, C_s, R_p, C_p", positive=True)
V, I = sp.symbols("V, I")

Zs = Rs + sp.I*omeg*Ls
Yp = 1/Rp + sp.I*omeg*Cp
Ys = sp.I*omeg*Cs

I = V/2*Yp + Ys*V+V/Zs
Zdiff = V/I

Q = sp.im(Zdiff)/sp.re(Zdiff)
Q = sp.simplify(sp.expand(Q))
sp.print_latex(Q)

\frac{R_{p} \omega \left(- C_{p} L_{s}^{2} \omega^{2} - C_{p} R_{s}^{2} - 2 C_{s} L_{s}^{2} \omega^{2} - 2 C_{s} R_{s}^{2} + 2 L_{s}\right)}{L_{s}^{2} \omega^{2} + 2 R_{p} R_{s} + R_{s}^{2}}


From the expression we obtained
$$
Q = \frac{R_{p} \omega \left(- C_{p} L_{s}^{2} \omega^{2} - C_{p} R_{s}^{2} - 2 C_{s} L_{s}^{2} \omega^{2} - 2 C_{s} R_{s}^{2} + 2 L_{s}\right)}{L_{s}^{2} \omega^{2} + 2 R_{p} R_{s} + R_{s}^{2}}
$$
we can factor the numerator
$$
Q = \frac{R_{p} \omega \left(2 L_{s} - (C_{p} + 2C_s) (L_{s}^{2} \omega^{2} + R_s^2) \right)}{L_{s}^{2} \omega^{2} + 2 R_{p} R_{s} + R_{s}^{2}},
$$
and reorder
$$
Q = \frac{\omega L_{s}}{R_s}\cdot\frac{ 2R_{p}  \left(1  - \frac{R_s^2\frac{C_{p} + 2C_s}{2}}{L_s} - \omega^{2}L_{s}\frac{C_{p} + 2C_s}{2} \right)}{2R_{p}+\left[\left(\frac{L_{s} \omega}{R_s}\right)^{2} +1\right]R_{s}},
$$
to see that in order to match the $Q$ factor expression in the paper, we must define
$$
C_\mathrm{tot} = C_p + 2C_s
$$

## Geometric programming

In [6]:
# Turn sympy expressions into functions
k6_func = sp.lambdify((k2, k4, k5, omeg), k6, modules="math")
k7_func = sp.lambdify((k2, k4, k5, omeg), k7, modules="math")

In [7]:

from gpkit import Variable, Model, ureg
import numpy as np

# Parameters
mu_0 = Variable("\\mu_0", 4*np.pi*1e-7, "H/m", "Magnetic permeability of free space", constant=True)
omega = Variable("\\omega", 2*np.pi*2.5e9, "rad/s", "Operating angular frequency", constant=True)
e_0= Variable("e_{0}", 8.854187e-12, "F/m", "Electric permittivity of free space", constant=True)

beta = 1.33e-3#Variable("beta", 1.33e-3, "-", "Beta coefficient for octagonal geommetry", constant=True)
alpha_1 = -1.21 #Variable("alpha_1", -1.21,  "-", "Alpha coefficient for outter diameter in octagonal geommetry", constant=True)
alpha_2 = -0.163 #Variable("alpha_2", -0.163, "-", "Alpha coefficient for track width in octagonal geommetry", constant=True)
alpha_3 = 2.43 #Variable("alpha_3",  2.43,  "-", "Alpha coefficient for average diameter in octagonal geommetry", constant=True)
alpha_4 = 1.75 #Variable("alpha_4",  1.75,  "-", "Alpha coefficient for number of turns in octagonal geommetry", constant=True)
alpha_5 = -0.049 #Variable("alpha_5", -0.049, "-", "Alpha coefficient for track spacing in octagonal geommetry", constant=True)

L_req = Variable("L_{req}", 10, "nH", "Required inductance in nano Henries", constant=True)
Q_min = Variable("Q_{min}", "-", "Minimum quality factor", positive=True)
omega_sr_min = Variable("\\omega_{sr,min}", 2*np.pi*7e9, "rad/s", "Minimum self-resonance frequency", constant=True)

sigma_tm2 = Variable("\\sigma_{tm2}", 30.3e6, "S/m", "Conductivity of TopMetal2 for ihp SG13G2", constant=True)
t_tm2 = Variable("t_{m2}", 3e-6, "m", "Thickness of TopMetal2 for ihp SG13Gw", constant=True)

e_ox = Variable("e_{ox}", 4.1*8.854187e-12, "F/m", constant=True)
t_ox = Variable("t_{ox}", 2.8e-6 + 0.85e-6 + 0.54e-6 + 0.54e-6 + 0.54e-6 + 0.54e-6 + 0.64e-6 , "m", "Oxide thickness between substrate and TopMetal2", constant=True)

sigma_tm1tm2=Variable("\\sigma_{TM1-TM2}", 3.143e6, "S/m", "Conductivity of TopVia2 for ihp SG13G2", constant=True)
t_ox_tm1tm2 = Variable("t_{ox,TM1-TM2}", 2.8e-6, "m", "Oxide thickness between TopMetal1 and TopMetal2", constant=True)

t_sub = Variable("t_{sub}", 180e-6, "m", "Substrate thickness for 200um back lapping option", constant=True)
e_sub = Variable("e_{sub}", 11.9*8.854187e-12, "F/m", constant=True)
sigma_sub = Variable("\\sigma_{sub}", 2, "S/m", "Conductivity of substrate for ihp SG13G2", constant=True)

w_min = Variable("w_{min}", 2e-6, "m", "Minimum TopMetal2 width", constant=True)
s_min = Variable("s_{min}", 2e-6, "m", "Minimum TopMetal2 separation", constant=True)
A_max = Variable("A_max", (250e-6)**2, "m^2", "Maximum TopMetal2 area", constant=True)

# Variables
d_out = Variable("d_{out}", "m", "Outter diameter of the inductor", positive=True)
d_avg = Variable("d_{avg}", "m", "Average diameter of the inductor", positive=True)

w = Variable("w", "m", "Metal track width", positive=True)
s = Variable("s", "m", "Spacing between metal tracks", positive=True)
n = Variable("n", "-", "Number of turns", positive=True)

# Monomials
skin_depth_tm1 = np.sqrt(2/(omega.value * mu_0.value * sigma_tm2.value))
l = 8 * d_avg * n/ (1 + 2**0.5) # Conductor length (8 sides of an octagon with a span of d_avg)

if (skin_depth_tm1 < t_tm2.value):
    print(f"Skin depth is {skin_depth_tm1.to("um")}")
    k_1 = 1 / (sigma_tm2 * skin_depth_tm1*(1 - np.exp(-t_tm2.value/skin_depth_tm1)))
else:
    print("No skin effect")
    k_1 = 1 / (sigma_tm2 * t_tm2)  # No skin effect occurs
k_2 = (e_ox )/(2 * t_ox)
k_3 = (e_ox)/(t_ox_tm1tm2)
k_4 = e_sub / (2 * t_sub)
k_5 = 2 * t_sub / sigma_sub

# Vias
## NOTE: This should be adjusted if other layers are used. This values are for TopVia1
a = Variable("a", 0.9e-6, "m", "TopVia1 width", constant=True)
b = Variable("b", 1.06e-6, "m", "TopVia1 width", constant=True)
c = Variable("c", 0.5e-6, "m", "TopVia1 width", constant=True)
skin_depth_via = np.sqrt(2/(omega.value * mu_0.value * sigma_tm1tm2.value))
if (skin_depth_via < a.value):
    print(f"Skin depth in via is {skin_depth_via.to("um")}")
    k_8 = 2 *t_ox_tm1tm2.value / (sigma_tm1tm2.value * a.value * skin_depth_via.value*(1 - np.exp(-a.value/skin_depth_via)) * (a.value + b.value)**2)
else:
    print("No skin effect in via")
    k_8 = 2 * t_ox_tm1tm2.value * (a.value + b.value)**2 / (sigma_tm1tm2.value * a.value * a.value)

# Design variables

R_m  = k_1 * l / w # Metal resistance
C_ox = k_2 * l * w
C_s  = k_3 * n * w**2  # should be n-1
C_si = k_4 * l * w
R_si = k_5 / (l * w)
R_v  = k_8.to("ohm*m^2").magnitude * n * (w/Variable("m", 1, "m", constant=True))**-2 * Variable("ohm", 1, "ohm", constant=True) # Approximates n-1 to n (not the best solution but works)


# L must be adjusted since the paper uses dimensionless units
um = Variable("um", 1e-6, "m", constant=True)

L = beta * (d_out/um)**alpha_1 \
        * (w/um)**alpha_2 \
        * (d_avg/um)**alpha_3 \
        * n**alpha_4 \
        * (s/um)**alpha_5 \
        * Variable("nH", 1e-9, "H", constant=True)

# Equivalent model
F_per_m2 = Variable("F_per_m2", 1, "F/m^2", constant=True)
ohm_m2   = Variable("ohm_m2", 1, "ohm*m^2", constant=True)

k_6 = k6_func(k_2.to("F/m^2").value.magnitude, k_4.to("F/m^2").value.magnitude, k_5.to("ohm*m^2").value.magnitude, omega.value.magnitude) * ohm_m2
k_7 = k7_func(k_2.to("F/m^2").value.magnitude, k_4.to("F/m^2").value.magnitude, k_5.to("ohm*m^2").value.magnitude, omega.value.magnitude) * F_per_m2


R_p = k_6/(l * w) 
C_p = k_7 * l * w 

Skin depth is 1.8286425166069231 micrometer
No skin effect in via


In [8]:
objective = Q_min**-1

# Normalized variables
R_s = Variable("R_{tot}", "ohm", positive=True)
C_tot = C_p + 2*C_s
rho = omega * L / R_s  # inductive reactance
gamma = omega**2 * L * C_tot/2  # Capacitive loading
delta = R_s**2 * C_tot/2 / L  # Q factor normalization

k_sr = omega_sr_min / omega

constraints = [
    L == L_req,  # Required inductance
    R_s >= R_m + R_v, # Total series resistance

    # normalized Q constraint
    Q_min * (2*R_p + (rho**2 + 1)*R_s) / (rho * 2*R_p) + delta + gamma <= 1,

    # self resonance
    delta/2 + gamma/2 <= 1,

    # minimum SRF
    k_sr**2 * gamma/2 + delta/2 <= 1,
    
    # PDK dimensional constraints
    w >= w_min,
    s >= s_min,
    d_out**2 <= A_max,

    #Geometry constriants
    d_avg + n*s + n*w <= d_out,
]

m = Model(objective, constraints)

#m.debug()
sol = m.solve()
print(sol.table())

Using solver 'cvxopt'
 for 7 free variables
  in 11 posynomial inequalities.
Solving took 0.0921 seconds.

          ┃┓
     Cost╺┫┃
 (0.0895) ┃┣╸Q_{min}^-1
          ┃┛ (0.0895)



       ┃┓
       ┃┃
       ┃┣╸d_{out} ≥ d_{avg} + n·s + n·w
       ┃┛
       ┃┣╸Q_{min}·2·0.000222·ohm_m2/(8·d_{avg}·n/2.41·w) + ((\omega·0.00133·…
       ┃┛
       ┃┣╸nH = 1e-09H
       ┃┛
       ┃┣╸um = 1e-06m
 Model╺┫┛
       ┃┣╸R_{tot} ≥ 1/(\sigma_{tm2}·1.83e-06 [m·s⁰⋅⁵/H⁰⋅⁵/rad⁰⋅⁵/S⁰⋅⁵]·0.806…
       ┃┛
       ┃┣╸\sigma_{tm2} = 3.03e+07S/m
       ┃┛
       ┃┣╸\omega = 1.57e+10rad/s
       ┃┣╸A_max = 6.25e-08m²
       ┃┣╸A_max ≥ d_{out}²
       ┃┣╸L_{req} = 0.00133·(d_{out}/um)^-1.21·(w/um)^-0.163·(d_{avg}/um)^2.…
       ┃┣╸[9 terms]
       ┃┛


Free Variables
--------------
Q_{min} : 11.17          Minimum quality factor
R_{tot} : 11.55      [Ω]
d_{avg} : 0.0001694  [m] Average diameter of the inductor
d_{out} : 0.00025    [m] Outter diameter of the inductor
      n : 7.482          Number of turns
   

In [9]:
print(f"""
L={L.sub(sol["variables"]).value.to("nH")}
w={w.sub(sol["variables"]).value.to("um")}
s={s.sub(sol["variables"]).value.to("um")}
l={l.sub(sol["variables"]).value.to("um")}
n={n.sub(sol["variables"]).value} turns
d_out={d_out.sub(sol["variables"]).value.to("um")}
d_in={(d_avg.sub(sol["variables"]).value - (n.sub(sol["variables"]).value -1)*s.sub(sol["variables"]).value.to("um") - n.sub(sol["variables"]).value * w.sub(sol["variables"]).value).to("um") }
d_avg={d_avg.sub(sol["variables"]).value.to("um")}
C_si={C_si.sub(sol["variables"]).value.to("fF")}
C_ox={C_ox.sub(sol["variables"]).value.to("fF")}
C_s={C_s.sub(sol["variables"]).value.to("fF")}
C_p={C_p.sub(sol["variables"]).value.to("fF")}
R_si={R_si.sub(sol["variables"]).value.to("kohm")}
R_m={R_m.sub(sol["variables"]).value.to("ohm")}
R_v={R_v.sub(sol["variables"]).value.to("ohm")}
R_s={R_s.sub(sol["variables"]).value.to("ohm")}
R_p={R_p.sub(sol["variables"]).value.to("kohm")}
""")


L=10.000000022614488 nanohenry
w=8.766409765415567 micrometer
s=2.0000000110989804 micrometer
l=4201.1874015883595 micrometer
n=7.482306830007374 turns
d_out=250.0000008182415 micrometer
d_in=90.88487521388437 micrometer
d_avg=169.44245660825837 micrometer
C_si=10.779210969841266 femtofarad
C_ox=103.64220826972644 femtofarad
C_s=7.455101534148378 femtofarad
C_p=10.964794961135865 femtofarad
R_si=4.887408994721235 kiloohm
R_m=10.72937470910592 ohm
R_v=0.8227402599625634 ohm
R_s=11.552114709428169 ohm
R_p=6.034094539498722 kiloohm



In [10]:
L_val  = L.sub(sol["variables"]).value
Ctot_val = C_tot.sub(sol["variables"]).value
Rs_val = R_s.sub(sol["variables"]).value
Rv_val = R_v.sub(sol["variables"]).value

omega_sr = np.sqrt(2/(L_val * Ctot_val) - Rs_val**2/L_val**2)
f_sr = omega_sr/(2*np.pi)

print("Self resonance frequency =", f_sr.to("GHz"))

Q_min_val = Q_min.sub(sol["variables"]).value
print("Q factor =", Q_min_val)

Self resonance frequency = 13.991280773765782 gigahertz
Q factor = 11.172937743800432


In [11]:
def imag(q):
    return q.magnitude.imag * q.units

def real(q):
    return q.magnitude.real * q.units

Cp_val = C_p.sub(sol["variables"]).value.to("fF")
Rp_val = R_p.sub(sol["variables"]).value.to("kohm")
Cs_val = C_s.sub(sol["variables"]).value.to("fF")

Zs_val = Rs_val + Rv_val + 1j*omega.value*L_val
Yp_val = 1/Rp_val + 1j*omega.value*Cp_val
Ys_val = 1j*omega.value*Cs_val

V = 1 * ureg("V")
I = V/2*Yp_val + Ys_val*V+V/Zs_val

Zdiff_val = (V/I).to("ohm")
Ldiff_val = imag(Zdiff_val) / omega.value
Ldiff_val = Ldiff_val.to("H")
Rdiff_val = real(Zdiff_val)
Qdiff_val = imag(Zdiff_val)/real(Zdiff_val)
print(f"""Zdiff={Zdiff_val}
Ldiff={Ldiff_val}
Rdiff={Rdiff_val}
Qdiff={Qdiff_val}
""")

Zdiff=(15.363054585715938+161.84137821795412j) ohm
Ldiff=1.030314213607696e-08 henry
Rdiff=15.363054585715938 ohm
Qdiff=10.534453113798664 dimensionless



# One port inductors
When one of the terminals of the inductor is grounded, the equivalent circuit consists of $R_p$, $C_\mathrm{tot}$ and $L_S + R_S$ in parallel, where $C_\mathrm{tot}=C_s + C_p$. This changes the expressions for the $Q$ factor and for self-resonance $\omega_\mathrm{sr}$, but everything else stays the same.

In [12]:
objective = Q_min**-1

# Normalized variables
R_s = Variable("R_{s}", "ohm", positive=True)
C_tot = C_p + C_s
rho = omega * L / R_s  # inductive reactance
gamma = omega**2 * L * C_tot  # Capacitive loading
delta = R_s**2 * C_tot / L  # Q factor normalization

k_sr = omega_sr_min / omega

constraints = [
    L == L_req,  # Required inductance
    R_s >= R_m + R_v, # Total series resistance

    # normalized Q constraint
    Q_min * (R_p + (rho**2 + 1)*R_s) / (rho * R_p) + delta + gamma <= 1,

    # self resonance
    delta/2 + gamma/2 <= 1,

    # minimum SRF
    k_sr**2 * gamma/2 + delta/2 <= 1,
    
    # PDK dimensional constraints
    w >= w_min,
    s >= s_min,
    d_out**2 <= A_max,

    #Geometry constriants
    d_avg + n*s + n*w <= d_out,
]

m = Model(objective, constraints)

#m.debug()
sol = m.solve()
print(sol.table())

Using solver 'cvxopt'
 for 7 free variables
  in 11 posynomial inequalities.
Solving took 0.0128 seconds.

         ┃┓
    Cost╺┫┃
 (0.104) ┃┣╸Q_{min}^-1
         ┃┛ (0.104)



       ┃┓
       ┃┃
       ┃┣╸d_{out} ≥ d_{avg} + n·s + n·w
       ┃┛
       ┃┓
       ┃┣╸Q_{min}·0.000222·ohm_m2/(8·d_{avg}·n/2.41·w) + ((\omega·0.00133·(d…
       ┃┛
       ┃┣╸nH = 1e-09H
       ┃┛
 Model╺┫┣╸um = 1e-06m
       ┃┛
       ┃┓
       ┃┃
       ┃┃
       ┃┃
       ┃┣╸[15 terms]
       ┃┃
       ┃┃
       ┃┃
       ┃┛


Free Variables
--------------
Q_{min} : 9.615          Minimum quality factor
  R_{s} : 11.81      [Ω]
d_{avg} : 0.0001773  [m] Average diameter of the inductor
d_{out} : 0.00025    [m] Outter diameter of the inductor
      n : 6.995          Number of turns
      s : 2e-06      [m] Spacing between metal tracks
      w : 8.388e-06  [m] Metal track width

Fixed Variables
---------------
          A_max : 6.25e-08   [m²]    Maximum TopMetal2 area
       F_per_m2 : 1          [F/m²]
   

In [13]:
print(f"""
L={L.sub(sol["variables"]).value.to("nH")}
w={w.sub(sol["variables"]).value.to("um")}
s={s.sub(sol["variables"]).value.to("um")}
l={l.sub(sol["variables"]).value.to("um")}
n={n.sub(sol["variables"]).value} turns
d_out={d_out.sub(sol["variables"]).value.to("um")}
d_in={(d_avg.sub(sol["variables"]).value - (n.sub(sol["variables"]).value -1)*s.sub(sol["variables"]).value.to("um") - n.sub(sol["variables"]).value * w.sub(sol["variables"]).value).to("um") }
d_avg={d_avg.sub(sol["variables"]).value.to("um")}
C_si={C_si.sub(sol["variables"]).value.to("fF")}
C_ox={C_ox.sub(sol["variables"]).value.to("fF")}
C_s={C_s.sub(sol["variables"]).value.to("fF")}
C_p={C_p.sub(sol["variables"]).value.to("fF")}
C_tot={C_tot.sub(sol["variables"]).value.to("fF")}
R_si={R_si.sub(sol["variables"]).value.to("kohm")}
R_m={R_m.sub(sol["variables"]).value.to("ohm")}
R_v={R_v.sub(sol["variables"]).value.to("ohm")}
R_s={R_s.sub(sol["variables"]).value.to("ohm")}
R_p={R_p.sub(sol["variables"]).value.to("kohm")}
""")


L=10.00000000044531 nanohenry
w=8.388090005111987 micrometer
s=2.0000000003547505 micrometer
l=4110.616489272058 micrometer
n=6.9952650437541815 turns
d_out=249.99999988803137 micrometer
d_in=106.66511735346667 micrometer
d_avg=177.33256023972558 micrometer
C_si=10.091673531378854 femtofarad
C_ox=97.03152975255807 femtofarad
C_s=6.381236684698873 femtofarad
C_p=10.265420298005406 femtofarad
C_tot=16.64665698270428 femtofarad
R_si=5.220384159890858 kiloohm
R_m=10.971550421241162 ohm
R_v=0.8401344244619562 ohm
R_s=11.811684828152801 ohm
R_p=6.445192450090775 kiloohm



In [14]:
L_val  = L.sub(sol["variables"]).value
Ctot_val = C_tot.sub(sol["variables"]).value
Rs_val = R_s.sub(sol["variables"]).value
Rv_val = R_v.sub(sol["variables"]).value

omega_sr = np.sqrt(2/(L_val * Ctot_val) - Rs_val**2/L_val**2)
f_sr = omega_sr/(2*np.pi)

print("Self resonance frequency =", f_sr.to("GHz"))

Q_min_val = Q_min.sub(sol["variables"]).value
print("Q factor =", Q_min_val)

Self resonance frequency = 17.444012802594482 gigahertz
Q factor = 9.615310243172342


# Inductors for LC resonators
Designing a tuned $LC$ tank is a common problem in electronics engineering. A one port inductor is used and an additional capacitance $C_\mathrm{ad}$ is connected in parallel to select the operating frequency.

The tank's inductance is given by a posynomial
$$
L_\mathrm{tank} = \left[1 + \left(\frac{R_s}{L_s\omega}\right)^2 \right] L_s
$$
The tank's capacitance is also given by a posynomial
$$
C_\mathrm{tank} = C_\mathrm{ad} + C_\mathrm{tot}
$$
The tank's resistance is given by
$$
R_\mathrm{tank} = R_p || R_{s,p} = \left(\frac{1}{R_p} + \frac{1}{R_{s,p}}\right)^{-1},
$$
where $R_{s,p}$ is the parallel equivalent of $R_s$. Even though this expression is not a polynomial, whenever $Q_\mathrm{tank}>1.5$, $R_{s,p}$ can be approximated by a monomial
$$
R_{s,p} = \left[1 + \left(\frac{L_s\omega}{R_s}\right)^2 \right]R_s \approx \frac{(L_s\omega)^2}{R_s}.
$$
The quality factor $Q_\mathrm{tank}$ is simply $R_\mathrm{tank}/(\omega_\mathrm{res}L_\mathrm{tank})$ and the inverse of both $R_\mathrm{tank}$ and $L_\mathrm{tank}$ are posynomial functions of desing variables, the inverse of $Q_\mathrm{tank}$ is a posynomial and $Q_\mathrm{tank}$ can be maixmized or a minimum required value imposed.

In [23]:
C_pad = Variable("C_ad", 38.98, "fF", "Pad capacitance", constant=True, positive=True) 
C_ss1 = Variable("C_ss1", 41, "fF", "Total source capacitance in M1", constant=True, positive=True)
C_dd2 = Variable("C_dd2", 29.57, "fF", "Total drain capacitance in M2", constant=True, positive=True)
C_ad = C_pad + C_dd2

In [26]:
Q_tankmin = Variable("Q_{tankmin}", 1.4, "-", "Minimum tank quality factor", constant=True)
R_s = Variable("R_{tot}", "ohm", positive=True)
C_tot = C_p + C_s

L_tank = (1 + (R_s / (L * omega))**2) * L
C_tank = C_ad + C_tot
inv_R_tank = 1/R_p +   R_s/(L * omega)**2

objective = inv_R_tank

constraints = [
    L_tank * C_tank <= omega**(-2),
    omega * L_tank * inv_R_tank <= 1/Q_tankmin,

    R_s >= R_m + R_v, # Total series resistance
    
    # PDK dimensional constraints
    w >= w_min,
    s >= s_min,
    #d_out**2 <= A_max,
    #n <= 5,

    #Geometry constriants
    d_avg + n*s + n*w <= d_out,
]

m = Model(objective, constraints)

#m.debug()
sol = m.solve()
#print(sol.summary())

Using solver 'cvxopt'
 for 6 free variables
  in 7 posynomial inequalities.
Solving took 0.0106 seconds.


In [27]:
print(f"""
L={L.sub(sol["variables"]).value.to("nH")}
w={w.sub(sol["variables"]).value.to("um")}
s={s.sub(sol["variables"]).value.to("um")}
l={l.sub(sol["variables"]).value.to("um")}
n={n.sub(sol["variables"]).value} turns
d_out={d_out.sub(sol["variables"]).value.to("um")}
d_in={(d_avg.sub(sol["variables"]).value - (n.sub(sol["variables"]).value -1)*s.sub(sol["variables"]).value.to("um") - n.sub(sol["variables"]).value * w.sub(sol["variables"]).value).to("um") }
d_avg={d_avg.sub(sol["variables"]).value.to("um")}
C_si={C_si.sub(sol["variables"]).value.to("fF")}
C_ox={C_ox.sub(sol["variables"]).value.to("fF")}
C_s={C_s.sub(sol["variables"]).value.to("fF")}
C_p={C_p.sub(sol["variables"]).value.to("fF")}
C_tot={C_tot.sub(sol["variables"]).value.to("fF")}
C_ad={C_ad.sub(sol["variables"]).value.to("fF")}
R_si={R_si.sub(sol["variables"]).value.to("kohm")}
R_m={R_m.sub(sol["variables"]).value.to("ohm")}
R_v={R_v.sub(sol["variables"]).value.to("ohm")}
R_s={R_s.sub(sol["variables"]).value.to("ohm")}
R_p={R_p.sub(sol["variables"]).value.to("kohm")}
L_tank={L_tank.sub(sol["variables"]).value.to("nH")}
C_tank={C_tank.sub(sol["variables"]).value.to("fF")}
R_tank={(1/inv_R_tank.sub(sol["variables"]).value).to("kohm")}
Q_tank={1/(omega * L_tank * inv_R_tank).sub(sol["variables"]).value}
""")


L=52.13249850438291 nanohenry
w=2.5525501331824287 micrometer
s=2.000005802337496 micrometer
l=9444.620764349862 micrometer
n=11.0595393123253 turns
d_out=308.06028811646286 micrometer
d_in=209.36197071321521 micrometer
d_avg=257.7111362513202 micrometer
C_si=7.055892497530962 femtofarad
C_ox=67.84246841479838 femtofarad
C_s=0.9342435609255995 femtofarad
C_p=7.17737269636009 femtofarad
C_tot=8.11161625728569 femtofarad
C_ad=68.55 femtofarad
R_si=7.466442079217467 kiloohm
R_m=82.83890864274554 ohm
R_v=14.343635664070922 ohm
R_s=97.18253817274932 ohm
R_p=9.218221235085997 kiloohm
L_tank=52.86672183702681 nanohenry
C_tank=76.66161625728567 femtofarad
R_tank=3.9463014761747153 kiloohm
Q_tank=4.752126593362246



In [28]:
L_tank_val  = L_tank.sub(sol["variables"]).value
C_tank_val = C_tank.sub(sol["variables"]).value

(1/(2*np.pi*np.sqrt(L_tank_val * C_tank_val))).to("GHz")

<Quantity(2.49999969, 'gigahertz')>